In [1]:
from dotenv import load_dotenv
import os
import requests

In [2]:
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from numpy import cos
from math import log
from scipy.interpolate import interp1d
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
from mpl_toolkits.mplot3d import Axes3D
from scipy.optimize import minimize_scalar
from scipy.optimize import minimize
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import plotly.graph_objects as go
import plotly.express as px
from scipy.optimize import fsolve
from sympy import symbols, Eq, solve

In [3]:
import requests

climate_api = "https://api.mosqlimate.org/api/datastore/climate"

page = 1 # total amount of pages is returned in the request
per_page = 100
pagination = f"?page={page}&per_page={per_page}&"
filters = "start=%s&end=%s" % ("2016-01-01", "2024-12-31")

resp = requests.get(climate_api + pagination + filters)

load_dotenv(encoding="utf-16")

api_key = os.getenv("MOSQLIMATE_API_KEY")

headers = {
    "X-UID-Key": api_key
}

# Or you can add a geocode to the filters
geocode = 1302603
resp = requests.get(
    climate_api + 
    pagination + 
    filters +
    f"&geocode={geocode}",
    headers=headers
)

items = resp.json()["items"] # JSON data in dict format
#total_items = resp.json()["total_items"] # JSON data in dict format
resp.json()["pagination"] # Pagination*

{'items': 100,
 'total_items': 3288,
 'page': 1,
 'total_pages': 33,
 'per_page': 100}

In [4]:
#### Getting the full data:

pages = np.arange(1, 34) # total amount of pages is returned in the request
per_page = 100
for page in pages:
    pagination = f"?page={page}&per_page={per_page}&"
    filters = "start=%s&end=%s" % ("2016-01-01", "2024-12-31")

    resp = requests.get(climate_api + pagination + filters)

# Or you can add a geocode to the filters
geocode = 1302603
resp = requests.get(
    climate_api + 
    pagination + 
    filters +
    f"&geocode={geocode}",
    headers=headers
)

items = resp.json()["items"] # JSON data in dict format
#total_items = resp.json()["total_items"] # JSON data in dict format
resp.json()["pagination"] # Pagination*

{'items': 88,
 'total_items': 3288,
 'page': 33,
 'total_pages': 33,
 'per_page': 100}

In [11]:
total_pages = 33 

start_date = "2016-01-01"
end_date = "2024-12-31"

geocode = 1302603

all_items = []  # List to store all items

# Loop through each page
for page in range(1, total_pages + 1):
    pagination = f"?page={page}&per_page={per_page}&"
    filters = f"start={start_date}&end={end_date}&geocode={geocode}"
    
    resp = requests.get(climate_api + pagination + filters, headers=headers)
    
    # Check if the request was successful
    if resp.status_code == 200:
        items = resp.json().get("items", [])  # Get items, default to an empty list if key not found
        all_items.extend(items)  # Append the items from this page to the list
    else:
        print(f"Failed to retrieve data for page {page}")
        break  # Break the loop in case of a failed request

full_data_df = pd.DataFrame(all_items)
full_data_df

,date,geocodigo,epiweek,temp_min,temp_med,temp_max,precip_min,precip_med,precip_max,precip_tot,pressao_min,pressao_med,pressao_max,umid_min,umid_med,umid_max
0,2016-01-01,1302603,201552,25.2045,26.9742,28.7045,0.2092,3.0621,6.9870,24.4965,0.9865,0.9892,0.9912,73.5599,83.5514,91.6022
1,2016-01-02,1302603,201552,24.7094,25.7935,26.7381,0.0388,7.8986,17.6067,63.1887,0.9880,0.9899,0.9922,86.0326,90.3434,95.2309
2,2016-01-03,1302603,201601,24.1877,25.8707,28.2754,1.1520,7.2728,20.6599,58.1821,0.9870,0.9897,0.9917,79.3838,90.4124,97.3460
3,2016-01-04,1302603,201601,24.2683,26.5175,29.6513,0.1548,6.1504,12.8499,49.2032,0.9866,0.9893,0.9914,72.0116,87.7519,96.4379
4,2016-01-05,1302603,201601,25.3694,27.0980,29.6138,0.0445,3.3259,8.8127,26.6074,0.9882,0.9898,0.9919,70.6908,85.9048,96.4212
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3283,2024-12-27,1302603,202452,24.3396,26.3144,29.6464,0.4375,4.4175,12.0925,35.3399,0.9873,0.9892,0.9908,72.7873,88.5445,95.8023
3284,2024-12-28,1302603,202452,24.6078,26.4918,30.1530,0.8793,4.4093,13.8036,35.2744,0.9871,0.9898,0.9916,69.3192,88.2399,96.5652
3285,2024-12-29,1302603,202501,25.0180,26.4557,28.5187,0.3675,5.6817,16.5503,45.4533,0.9875,0.9896,0.9918,80.2842,89.0658,94.7284
3286,2024-12-30,1302603,202501,24.1221,25.7000,27.6957,0.1818,5.2545,16.7795,42.0363,0.9868,0.9894,0.9913,80.0111,89.6784,96.1786


In [13]:
# Saves the dataframe as a separate csv file to be used in different modules
full_data_df.to_csv("../data/climate_api_data_2016_2024.csv", index=False)

In [15]:
datetime_df = pd.DataFrame()
datetime_df['date'] = pd.to_datetime(full_data_df['date'])

# Extract the year from the 'date' column
datetime_df['year'] = datetime_df['date'].dt.year

# Count the occurrences of each year
year_counts = datetime_df['year'].value_counts().sort_index()

print(year_counts)

year
2016    366
2017    365
2018    365
2019    365
2020    366
2021    365
2022    365
2023    365
2024    366
Name: count, dtype: int64
